## Project: Superstore Sales Data Analysis


In [1]:
import pandas as pd
import  numpy as np
import os
import json
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import warnings 
import logging
warnings.filterwarnings('ignore')


# Configuration Display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

In [2]:
# Self test: 
logger.debug("debug message")
logger.info("Info message")
logger.error("This is error")

INFO: Info message
ERROR: This is error


### Configuration

In [3]:
# Configuration
# Define file path
RAW_DATA_PATH = 'E:/PYTHON/Projects/Sales_Analysis_Project/data/raw/superstore_data.csv'
PROCESSED_DATA_PATH = 'E:\PYTHON\Projects\Sales_Analysis_Project\data\processed'
OUTPUT_PATH = 'E:\PYTHON\Projects\Sales_Analysis_Project\output'

# Create directories if they don't exist
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
print("✅ Configuration complete!")

✅ Configuration complete!


In [5]:
# Load the Data
def load_superstore_data(filepath):
    if not os.path.exists(filepath):
        logger.error(f"❌ File not found: {filepath}")
        return None
    
    try:
        df = pd.read_csv(filepath, encoding='latin-1')
        logger.info(f'Successfully Loaded {len(df):,} rows from {filepath} ')
        logger.info(f"Columns: {len(df.columns)} columns found")
        return df
    except Exception as e:
        logger.error(f"❌ Error loading file: {e}")
        return None
    
# Load Data (Call fxn)
df_raw = load_superstore_data(RAW_DATA_PATH)


if df_raw is not None:
    print("First 5 rows of raw data:")
    display(df_raw.head())
    print(f"\nData shape: {df_raw.shape}")
    print(f"\n Columns: {df_raw.columns.tolist()}")


INFO: Successfully Loaded 9,994 rows from E:/PYTHON/Projects/Sales_Analysis_Project/data/raw/superstore_data.csv 
INFO: Columns: 21 columns found


First 5 rows of raw data:


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164



Data shape: (9994, 21)

 Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


Initial Data Exploration

In [6]:
def explore_data(df):
    print("="*60)
    print("DATA EXPLORATION REPORT")
    print("="*60)
    
    print(f"\n Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    
    print("\n Column Information: ")
    print("-"*40)
    for col in df.columns:
        dtype = df[col].dtype
        nulls = df[col].isnull().sum()
        unique = df[col].nunique()
        print(f" • {col:20s} | {str(dtype):10s} | Null: {nulls:4d} | Unique: {unique:5d}")

    print("\n Basic Statistics:")
    print("-"*40)
    display(df.describe())

    print("\n Missing Values Summery:")
    missing_data = df.isnull().sum()
    missing_data = missing_data[missing_data > 0].sort_values(ascending=False)
    if len(missing_data) > 0:
        for col, count in missing_data.items():
            print(f"• {col}: {count:,} missing ({count/len(df_raw)*100:.1f}%)")
    else:
        print("  ✅ No missing values found!")

if df_raw is not None:
    explore_data(df_raw)


DATA EXPLORATION REPORT

 Dataset Shape: 9994 rows x 21 columns

 Column Information: 
----------------------------------------
 • Row ID               | int64      | Null:    0 | Unique:  9994
 • Order ID             | str        | Null:    0 | Unique:  5009
 • Order Date           | str        | Null:    0 | Unique:  1237
 • Ship Date            | str        | Null:    0 | Unique:  1334
 • Ship Mode            | str        | Null:    0 | Unique:     4
 • Customer ID          | str        | Null:    0 | Unique:   793
 • Customer Name        | str        | Null:    0 | Unique:   793
 • Segment              | str        | Null:    0 | Unique:     3
 • Country              | str        | Null:    0 | Unique:     1
 • City                 | str        | Null:    0 | Unique:   531
 • State                | str        | Null:    0 | Unique:    49
 • Postal Code          | int64      | Null:    0 | Unique:   631
 • Region               | str        | Null:    0 | Unique:     4
 • Product ID 

,Row ID,Postal Code,Sales,Quantity,Discount,Profit
count,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,4997.500000,55190.379428,229.858001,3.789574,0.156203,28.656896
std,2885.163629,32063.693350,623.245101,2.225110,0.206452,234.260108
min,1.000000,1040.000000,0.444000,1.000000,0.000000,-6599.978000
25%,2499.250000,23223.000000,17.280000,2.000000,0.000000,1.728750
50%,4997.500000,56430.500000,54.490000,3.000000,0.200000,8.666500
75%,7495.750000,90008.000000,209.940000,5.000000,0.200000,29.364000
max,9994.000000,99301.000000,22638.480000,14.000000,0.800000,8399.976000



 Missing Values Summery:
  ✅ No missing values found!


Data Cleaning


In [ ]:
def clean_superstore_data(df, stats):
    df_clean = df.copy()
    initial_rows = len(df_clean)
    stats['rows-read'] = initial_rows

    print("Starting Data Cleaning Process...")
    print("-"*60)

    # step 1: Handle Missing Values
    print("Step 1: Handling Missing Values")
    before = len(df_clean)
    
    # Filling missing values fro critical columns
    df_clean['Postal Code'] = df_clean['Postal Code'].fillna('Unknown')
    df_clean['Ship Date'] = df_clean['Ship Date'].fillna(df_clean['Order Date'])

    # Drop rows with missing essential data
    essential_cols = ['Order ID', 'Customer ID', 'Product ID', 'Sales', 'Quantity']
    df_clean = df_clean.dropna(subset=essential_cols)

    after = len(df_clean)
    stats['rows_with_missing_values'] = before - after
    print(f" • Removed {stats['rows_with_missing_values']:,} rows with missing values")

    # Step 2: Fix Data Formats
    print("\n Step 2: Standardizing Date")
    def standardize_date(date_value):
        if pd.isna(date_value):
            return None
        try:
            dt = pd.to_datetime(date_value)
            return dt.strftime('%Y-%m-%d')
        except:
            return None
    


    

In [17]:
df_raw.isnull().sum().any()

np.False_

In [13]:
import pandas as pd
import numpy as np

# Create a sample dataset with 10 rows
data = {
    'Order ID': ['ORD-001', 'ORD-002', 'ORD-003', 'ORD-004', 'ORD-005',
                 'ORD-006', 'ORD-007', 'ORD-008', 'ORD-009', 'ORD-010'],
    'Customer': ['John', 'Sarah', 'Mike', 'Emma', 'David',
                 'Lisa', 'Tom', np.nan, 'Anna', 'Robert'],  # 1 missing
    'Sales': [100, 150, np.nan, 200, 175,  # 1 missing
              120, 180, 90, 210, 160],
    'Quantity': [2, 3, 1, np.nan, 2,  # 1 missing
                 1, 3, 2, 1, 2],
    'Profit': [20, 30, 10, 40, 25,
               15, 35, 18, 42, 28]
}

df_sample = pd.DataFrame(data)
print(df_sample)

stats = {}
cleaned_data = data.copy()
initial_rows = len(cleaned_data)
print("Initial_rows:", initial_rows)
stats['rows_read'] = initial_rows
print(stats)

  Order ID Customer  Sales  Quantity  Profit
0  ORD-001     John  100.0       2.0      20
1  ORD-002    Sarah  150.0       3.0      30
2  ORD-003     Mike    NaN       1.0      10
3  ORD-004     Emma  200.0       NaN      40
4  ORD-005    David  175.0       2.0      25
5  ORD-006     Lisa  120.0       1.0      15
6  ORD-007      Tom  180.0       3.0      35
7  ORD-008      NaN   90.0       2.0      18
8  ORD-009     Anna  210.0       1.0      42
9  ORD-010   Robert  160.0       2.0      28
Initial_rows: 5
{'rows_read': 5}
